Text & Image Preprocessing
- Preprocess captions (tokenization, lowercasing, stop-word handling, vocabulary building) using NLTK/spaCy
- Preprocess images (resizing, normalization, channel consistency check) using OpenCV/torchvision


## 1. Setup

Import libraries and define paths.

In [1]:
import os
import re
import json
import string
import pandas as pd
import numpy as np
import cv2
import torch
from torchvision import transforms
from collections import Counter

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

RAW_IMG_DIR = "../data/flickr8k/Images"
CAPTIONS_PATH = "../data/flickr8k/captions.txt"

PROCESSED_DIR = "../data/processed"
PROCESSED_IMG_DIR = os.path.join(PROCESSED_DIR, "images")
os.makedirs(PROCESSED_IMG_DIR, exist_ok=True)

print("Processed directory ready at:", PROCESSED_DIR)

[nltk_data] Downloading package punkt to C:\Users\Fast
[nltk_data]     Computer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Fast
[nltk_data]     Computer\AppData\Roaming\nltk_data...


Processed directory ready at: ../data/processed


[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Fast
[nltk_data]     Computer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 2. Text Preprocessing (Captions)

### 2.1 Load raw captions

In [2]:
captions_df = pd.read_csv(CAPTIONS_PATH)
print("Total caption rows:", len(captions_df))
captions_df.head()

Total caption rows: 40455


,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


### 2.2 Cleaning function

Steps applied to every caption:
1. Lowercase
2. Remove punctuation and digits
3. Tokenize (NLTK `word_tokenize`)
4. Remove stopwords 

In [3]:
stop_words = set(stopwords.words("english"))
translator = str.maketrans("", "", string.punctuation + string.digits)

def clean_and_tokenize(caption, remove_stopwords=False):
    caption = caption.lower().strip()
    caption = caption.translate(translator)
    caption = re.sub(r"\s+", " ", caption).strip()

    tokens = word_tokenize(caption)

    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]

    return tokens

# Quick test
sample = captions_df["caption"].iloc[0]
print("Original:", sample)
print("Tokenized (all tokens):", clean_and_tokenize(sample))
print("Tokenized (no stopwords):", clean_and_tokenize(sample, remove_stopwords=True))

Original: A child in a pink dress is climbing up a set of stairs in an entry way .
Tokenized (all tokens): ['a', 'child', 'in', 'a', 'pink', 'dress', 'is', 'climbing', 'up', 'a', 'set', 'of', 'stairs', 'in', 'an', 'entry', 'way']
Tokenized (no stopwords): ['child', 'pink', 'dress', 'climbing', 'set', 'stairs', 'entry', 'way']


### 2.3 Apply cleaning to the full caption set

In [4]:
captions_df["tokens"] = captions_df["caption"].apply(lambda c: clean_and_tokenize(c, remove_stopwords=False))
captions_df["tokens_no_stopwords"] = captions_df["caption"].apply(lambda c: clean_and_tokenize(c, remove_stopwords=True))

# The task requires stop-word handling to actually feed the cleaned/tokenized
# deliverable, so cleaned_caption and downstream vocab use tokens_no_stopwords.
captions_df["cleaned_caption"] = captions_df["tokens_no_stopwords"].apply(lambda t: " ".join(t))

captions_df.head()

,image,caption,tokens,tokens_no_stopwords,cleaned_caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...,"[a, child, in, a, pink, dress, is, climbing, u...","[child, pink, dress, climbing, set, stairs, en...",child pink dress climbing set stairs entry way
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .,"[a, girl, going, into, a, wooden, building]","[girl, going, wooden, building]",girl going wooden building
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .,"[a, little, girl, climbing, into, a, wooden, p...","[little, girl, climbing, wooden, playhouse]",little girl climbing wooden playhouse
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...,"[a, little, girl, climbing, the, stairs, to, h...","[little, girl, climbing, stairs, playhouse]",little girl climbing stairs playhouse
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...,"[a, little, girl, in, a, pink, dress, going, i...","[little, girl, pink, dress, going, wooden, cabin]",little girl pink dress going wooden cabin


### 2.4 Drop empty / too-short captions

In [5]:
before = len(captions_df)
captions_df = captions_df[captions_df["tokens_no_stopwords"].apply(len) >= 2].reset_index(drop=True)
after = len(captions_df)

print(f"Dropped {before - after} short/empty captions. Remaining: {after}")

Dropped 10 short/empty captions. Remaining: 40445


### 2.5 Build vocabulary

In [6]:
# Build vocabulary from stop-word-removed tokens, as required by the task spec
all_tokens = [tok for tokens in captions_df["tokens_no_stopwords"] for tok in tokens]
word_freq = Counter(all_tokens)

print("Total tokens (no stopwords):", len(all_tokens))
print("Unique words:", len(word_freq))
print("Top 10 most common words:", word_freq.most_common(10))

# Build vocabulary — keep words appearing at least MIN_FREQ times
MIN_FREQ = 2
vocab_words = [w for w, freq in word_freq.items() if freq >= MIN_FREQ]

special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]
vocab = special_tokens + sorted(vocab_words)

word2idx = {w: i for i, w in enumerate(vocab)}
# NOTE: JSON has no integer keys, so idx2word keys will be saved/reloaded as strings
# ("0", "1", ...). Cast to int when looking up by index after loading vocab.json,
# e.g. idx2word[str(predicted_idx)].
idx2word = {i: w for w, i in word2idx.items()}

print("Final vocabulary size (incl. special tokens):", len(vocab))

Total tokens (no stopwords): 251037
Unique words: 8660
Top 10 most common words: [('dog', 8136), ('man', 7265), ('two', 5638), ('white', 3940), ('black', 3832), ('boy', 3581), ('woman', 3402), ('girl', 3328), ('wearing', 3062), ('people', 2883)]
Final vocabulary size (incl. special tokens): 5095


### 2.6 Save cleaned captions and vocabulary

In [7]:
# Save cleaned captions (includes both full tokens and stop-word-removed tokens
# for transparency, plus the final cleaned_caption string used for the vocabulary)
captions_out_path = os.path.join(PROCESSED_DIR, "captions_cleaned.csv")
captions_df[["image", "caption", "tokens", "tokens_no_stopwords", "cleaned_caption"]].to_csv(
    captions_out_path, index=False
)

# Save vocabulary (idx2word keys become strings after JSON round-trip — see note above)
vocab_out_path = os.path.join(PROCESSED_DIR, "vocab.json")
with open(vocab_out_path, "w") as f:
    json.dump({"word2idx": word2idx, "idx2word": idx2word}, f, indent=2)

print("Saved cleaned captions to:", captions_out_path)
print("Saved vocabulary to:", vocab_out_path)

Saved cleaned captions to: ../data/processed\captions_cleaned.csv
Saved vocabulary to: ../data/processed\vocab.json


## 3. Image Preprocessing

In [8]:
sample_files = os.listdir(RAW_IMG_DIR)[:200]  # sample first 200 for a quick check
channel_counts = Counter()

for fname in sample_files:
    img = cv2.imread(os.path.join(RAW_IMG_DIR, fname))
    if img is None:
        channel_counts["unreadable"] += 1
        continue
    channel_counts[img.shape[2] if img.ndim == 3 else 1] += 1

print("Channel distribution in sample:", channel_counts)

Channel distribution in sample: Counter({3: 200})


### Define resize + normalize transform

In [9]:
IMG_SIZE = 224

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

print("Transform pipeline ready:", preprocess)

Transform pipeline ready: Compose(
    ToPILImage()
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


### Process and save all images

In [10]:
def load_and_fix_channels(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)  # forces 3-channel BGR read
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

image_files = os.listdir(RAW_IMG_DIR)
failed = []
processed_count = 0

for fname in image_files:
    img_path = os.path.join(RAW_IMG_DIR, fname)
    img = load_and_fix_channels(img_path)

    if img is None:
        failed.append(fname)
        continue

    tensor = preprocess(img)

    out_path = os.path.join(PROCESSED_IMG_DIR, fname.replace(".jpg", ".pt"))
    torch.save(tensor, out_path)
    processed_count += 1

print(f"Processed {processed_count} images successfully.")
print(f"Failed to process {len(failed)} images.")
if failed:
    print("Examples of failed files:", failed[:5])

Processed 8091 images successfully.
Failed to process 0 images.


### Sanity check — reload a processed image tensor

In [12]:
pt_files = [f for f in os.listdir(PROCESSED_IMG_DIR) if f.endswith(".pt")]

sample_pt = pt_files[0]

tensor = torch.load(
    os.path.join(PROCESSED_IMG_DIR, sample_pt)
)

print(tensor.shape)

torch.Size([3, 224, 224])
